# ETFs: Holdout Backtest

**Chapter 20 - Out-of-sample evaluation**

[`18_holdout_predictions`](18_holdout_predictions.ipynb) refitted the selected
configuration on the history before the holdout window and wrote its predictions over it.
This notebook trades them, with the concentration, the allocator, the risk overlay and the
cost assumption the rest of the case study used, and registers the result.

Nothing is chosen here. The predictions, the allocator, the concentration, the rebalance
cadence, the overlay and the charge all arrive fixed from earlier notebooks, and the only
thing this notebook decides is that they are applied unchanged. That is the whole design: a
holdout result is worth something exactly to the extent that no decision was made after
seeing it, and every knob left open here would be a decision.

The comparison to validation is printed but not interpreted. The validation figure is the
maximum of a ranking over more than a thousand backtests and the holdout figure is one
measurement over a much shorter window; what can be said about the gap between them is
[`20_strategy_analysis`](20_strategy_analysis.ipynb)'s subject, with the intervals to say
it.

**Prerequisites:** [`18_holdout_predictions`](18_holdout_predictions.ipynb).

**Scope:** one backtest. No selection, no comparison beyond a printed pair.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# ETF：保留測試集回測

**第 20 章——樣本外評估**

[`18_holdout_predictions`](18_holdout_predictions.ipynb) 已使用保留測試期間之前的歷史資料，重新擬合選定組態，並產生該期間的預測。本 Notebook 依案例研究其他部分所採用的集中程度、配置方法、風險覆蓋規則及成本假設，交易這些預測並登錄結果。

此處不做任何選擇。預測、配置方法、集中程度、重新平衡頻率、覆蓋規則及成本都已由先前 Notebook 固定，本 Notebook 唯一要決定的是原樣套用它們。這就是整體設計：保留測試結果的價值，完全取決於看過結果後沒有再做決策的程度；此處每個尚可調整的設定，都會變成一項新決策。

與驗證結果的比較會列印出來，但不在這裡解釋。驗證數值是超過一千次回測排名中的最大值，保留測試數值則是在短得多的期間內進行的一次衡量；兩者差距能說明什麼，是 [`20_strategy_analysis`](20_strategy_analysis.ipynb) 的主題，屆時也會提供據以判讀的區間。

**先備條件：**[`18_holdout_predictions`](18_holdout_predictions.ipynb)。

**範圍：**一次回測。不進行篩選；除列印一對數值外，也不進行比較。
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""ETFs: Holdout Backtest."""

import json
import sqlite3
import warnings

import polars as pl

warnings.filterwarnings("ignore")

from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.utils.backtest_loaders import get_backtest_config, load_backtest_prices_for
from case_studies.utils.backtest_presets import ensure_backtest_spec, strategy_view
from case_studies.utils.backtest_runner import resolved_allow_short_selling, run_backtest
from case_studies.utils.conformal import (
    compute_holdout_conformal_widths,
    ensure_conformal_calibration_identity,
    holdout_conformal_embargo_steps,
)
from case_studies.utils.registry import (
    backtest_run_status,
    read_predictions,
    training_hash_from_spec,
)
from case_studies.utils.strategy_analysis import resolve_solvent_carrier
from utils.paths import get_case_study_dir

In [2]:
CASE_STUDY_ID = "etfs"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
MAX_SYMBOLS = 0

In [3]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
bt_config = get_backtest_config(CASE_STUDY_ID)


def _registered_holdout_backtests(case_dir, prediction_hash):
    """The backtest hashes already registered against one holdout prediction set."""
    with sqlite3.connect(str(case_dir / "run_log" / "registry.db")) as conn:
        rows = conn.execute(
            "SELECT backtest_hash FROM backtest_runs WHERE prediction_hash = ? "
            "ORDER BY backtest_hash",
            (prediction_hash,),
        ).fetchall()
    return [backtest_hash for (backtest_hash,) in rows]

## 1. The configuration, and the predictions it produced on the holdout

The selected configuration is resolved the same way [`17_costs`](17_costs.ipynb) and
[`18_holdout_predictions`](18_holdout_predictions.ipynb) resolve it, so all three run the
same configuration by construction rather than by a hash copied between them.

Which holdout prediction set belongs to it is derived rather than searched for. Re-deriving the
holdout training specification reproduces the training identity 18 registered - the derivation is
deterministic and the identity covers it - so the prediction set is looked up by that identity
and the selected configuration's checkpoint. A search over holdout prediction sets would have to
guess which one belonged to this configuration.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 組態及其在保留測試集產生的預測

選定組態的解析方式，與 [`17_costs`](17_costs.ipynb) 及 [`18_holdout_predictions`](18_holdout_predictions.ipynb) 相同。因此，三者在設計上就會執行同一組態，而不必在 Notebook 之間複製雜湊值。

屬於該組態的保留測試集預測組合，是推導得出而非搜尋得出。重新推導保留測試集訓練規格，會重現第 18 份 Notebook 登錄的訓練識別項——推導程序具決定性，識別項也涵蓋該規格——因此可以依識別項與選定組態的檢查點查找預測組合。若改為搜尋保留測試集預測組合，就得猜測哪一組屬於這個組態。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
LABEL = carrier["label"]
validation_prediction_record = study.results.open(carrier["val_prediction_hash"]).registry_record()

holdout_spec = build_holdout_training_spec(
    study,
    study.results.open(carrier["training_hash"]).spec(),
    timeline=(
        pl.read_parquet(study.root / "labels" / f"{LABEL}.parquet")
        .get_column("timestamp")
        .unique()
        .sort()
        .to_list()
    ),
    case_study=CASE_STUDY_ID,
)
holdout_training_hash = training_hash_from_spec(holdout_spec)

with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
    match = conn.execute(
        """
        SELECT prediction_hash FROM prediction_sets
        WHERE split = 'holdout' AND training_hash = ?
          AND checkpoint_kind IS ? AND checkpoint_value IS ?
        """,
        (
            holdout_training_hash,
            validation_prediction_record["checkpoint_kind"],
            validation_prediction_record["checkpoint_value"],
        ),
    ).fetchone()
if match is None:
    raise RuntimeError(
        f"No holdout prediction set for training {holdout_training_hash}. Run "
        "18_holdout_predictions first; this notebook does not fit."
    )
HOLDOUT_PREDICTION_HASH = match[0]

print(f"Selected configuration: {carrier['val_backtest_hash']}  {carrier['config_name']} ({LABEL})")
print(f"Holdout training:   {holdout_training_hash}")
print(f"Holdout prediction: {HOLDOUT_PREDICTION_HASH}")

Selected configuration: 18c7a8419306  ols (fwd_ret_5d)
Holdout training:   5ff0fa98f34e
Holdout prediction: 0afd2e256128


## 2. Calibration, where the allocator needs it

An allocator that sizes by a conformal width is calibrated from errors the model has already
made, and on the holdout there are none to use: an error is usable only once the return it
measures has been realised, and every holdout return realises inside the window being evaluated.
Such a selected configuration takes its widths from the validation residuals of the validation
prediction set, which is what the allocator would have had standing at the start of the window.

The embargo matters for this case study's label and would not for every one. A residual
observed at `t` measures a return realising over `(t, t+21]` sessions, so the last residuals
of the validation span reach into the holdout window, and calibrating on them would size
holdout positions with holdout price information. The step count comes from the reviewed
table in `conformal.py`, which records the label horizon.

The branch is here rather than assumed away because the selected configuration can change. It
does not fire for the one this case study currently reports, which sizes by inverse volatility
over a declared window and needs no calibration - so the line below prints that rather than
staying silent, which is what tells a reader the branch was evaluated.

The widths themselves are NOT written here. Writing them replaces the artifact an already
registered run was sized by, and the replacement guard in section 3 can still refuse this run
afterwards - which would leave the registered holdout pointing at a calibration that no
longer existed. The write is below the guard.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 2. 配置方法需要時所用的校準

依共形預測區間寬度設定部位規模的配置方法，需要以模型已產生的誤差進行校準；但在保留測試期間開始時，沒有可用的保留測試誤差：只有所衡量的報酬率實現後，誤差才能使用，而每個保留測試報酬率都在受評估的期間內才實現。若選定組態使用這種方法，其區間寬度會取自驗證預測組合的驗證殘差；這正是配置方法在保留測試期間開始時原本可取得的資訊。

對本案例研究的標籤而言，禁運期很重要，但並非每個案例研究都如此。在 `t` 觀察到的殘差，衡量的是跨越 `(t, t+21]` 個交易時段才實現的報酬率，因此驗證期間末端的殘差會延伸到保留測試期間；若拿它們校準，就會使用保留測試期間的價格資訊設定保留測試部位規模。步數取自 `conformal.py` 中已審核的表格，其中記錄標籤的預測期間。

此處保留這個分支，而非假設它不會用到，因為選定組態可能改變。本案例研究目前報告的組態不會進入這個分支：它依指定視窗內的波動率倒數設定部位規模，不需要校準。因此，下方會明確印出這一點，而非保持沉默，讓讀者知道此分支已受檢查。

區間寬度本身**不會**在此處寫入。若寫入，就會取代已登錄執行結果原本據以設定部位規模的資料產物；而第 3 節的替換防護仍可能在之後拒絕本次執行，導致已登錄的保留測試結果指向不再存在的校準資料。寫入動作放在防護檢查之後。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
allocation = strategy_view(json.loads(carrier["spec_json"])).get("allocation") or {}
NEEDS_CALIBRATION = allocation.get("method") == "conformal_weighted"
embargo_steps = holdout_conformal_embargo_steps(CASE_STUDY_ID, LABEL) if NEEDS_CALIBRATION else 0
if NEEDS_CALIBRATION:
    print(f"Conformal configuration: embargo {embargo_steps} observation(s), widths written below.")
else:
    print(f"Allocator {allocation.get('method', 'equal_weight')!r} needs no calibration.")

Allocator 'hrp' needs no calibration.


## 3. The backtest

The strategy specification is the selected configuration's own, re-pointed at the holdout
prediction set and the holdout price window. Nothing else about it changes - the concentration,
the allocator, the risk overlay and the commission and slippage levels are the ones `setup.yaml`
declares and every validation number in this case study was net of.

The run registers under `stage='holdout'`, which the registry derives from the prediction set's
split rather than from anything asserted here - and that derivation takes precedence over the
risk block the selected configuration carries, which would otherwise file this as another risk
overlay.

**The window carries one backtest**, for the same reason 18 lets it carry one prediction
generation. 18's guard is on the model - the training identity and the checkpoint - and it
cannot see this one: a changed allocator, overlay, cost level or calibration produces the
same holdout predictions and a different result from them. Like 18's, this guard refuses
rather than offering a replacement, because deleting a result that has been seen does not
unsee it.

The test is the backtest hash, not a field-by-field comparison. Every input that changes the
result is in that hash by construction, and a guard naming fields instead has to be right
about all of them. The hash is resolved before anything runs, so nothing is evaluated on the
holdout before the question is answered, and it comes from `backtest_run_status` - the call
the runner itself makes - because asking it is the only way to be sure the guard and the
runner agree about identity. The run asserts they still agree afterwards, because a guard
that had quietly stopped predicting the hash would let everything through while looking
correct.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 3. 回測

策略規格取自選定組態本身，再將其指向保留測試集預測組合與保留測試期間的價格資料。其餘部分完全不變：集中程度、配置方法、風險覆蓋規則，以及佣金與滑價水準，都與 `setup.yaml` 指定的相同；本案例研究的每個驗證數值也都已扣除這些成本。

執行結果以 `stage='holdout'` 登錄；登錄系統會從預測組合所屬的資料切分推導此值，而非依據此處的宣稱。這項推導優先於選定組態所帶的風險設定區塊，否則這次執行會被歸檔成另一筆風險覆蓋規則回測。

**這段期間只允許一筆回測**，原因與第 18 份 Notebook 只允許一代預測相同。第 18 份的防護檢查針對模型——訓練識別項與檢查點——無法察覺此處的改動：若配置方法、覆蓋規則、成本水準或校準方式改變，保留測試集預測仍相同，卻會由此產生不同結果。與第 18 份的防護相同，這裡會拒絕替換，因為刪除已看過的結果，並不能讓人忘記它。

檢查依據是回測雜湊值，而非逐欄位比較。設計上，每項會改變結果的輸入都納入該雜湊值；若防護檢查逐一列出欄位，就必須保證所有欄位都正確。雜湊值會在任何執行開始前解析，因此在回答是否允許執行前，不會先對保留測試集做任何評估；它取自 `backtest_run_status`，也就是執行器本身會使用的呼叫，因為只有詢問它，才能確定防護檢查與執行器對識別項的認定一致。執行後還會確認兩者仍一致，因為若防護檢查悄悄不再能預測雜湊值，便會在看似正確的情況下放行所有執行。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
prices = load_backtest_prices_for(CASE_STUDY_ID, LABEL, split="holdout", max_symbols=MAX_SYMBOLS)
predictions = read_predictions(CASE_STUDY_ID, HOLDOUT_PREDICTION_HASH)
print(f"Prices: {len(prices):,} rows, {prices['symbol'].n_unique():,} funds")
print(f"Predictions: {predictions.height:,} rows, {predictions['timestamp'].n_unique():,} sessions")

spec = ensure_backtest_spec(
    CASE_STUDY_ID,
    bt_config,
    json.loads(carrier["spec_json"]),
    prices=prices,
    prediction_hash=HOLDOUT_PREDICTION_HASH,
    initial_cash=bt_config.initial_cash,
)
spec["chapter"] = "ch20"
# The embargo goes into the specification before anything hashes it. The widths are an input to
# this backtest and the embargo decides them, so two embargoes are two results and must not
# share an identity.
if NEEDS_CALIBRATION:
    spec = ensure_conformal_calibration_identity(spec, holdout_embargo_steps=embargo_steps)

spec["backtest_config"]["account"]["allow_short_selling"] = resolved_allow_short_selling(spec, None)
prospective_hash = backtest_run_status(CASE_STUDY_ID, HOLDOUT_PREDICTION_HASH, spec).backtest_hash
superseded_backtests = sorted(
    set(_registered_holdout_backtests(CASE_DIR, HOLDOUT_PREDICTION_HASH)) - {prospective_hash}
)
if superseded_backtests:
    raise RuntimeError(
        "the holdout window already carries a backtest of a different configuration: "
        + ", ".join(superseded_backtests)
        + f". This run would register {prospective_hash} and has not run. Same rule as "
        "18_holdout_predictions: discarding the earlier result would not undo having observed "
        "it, so there is no switch here. Leave the selection where it was, or retire the "
        "earlier evaluation through the registry's lifecycle."
    )

# The guard has passed, so this run will register and the widths it is sized by are the ones
# that belong beside this prediction set.
if NEEDS_CALIBRATION:
    widths = compute_holdout_conformal_widths(
        CASE_STUDY_ID,
        carrier["val_prediction_hash"],
        HOLDOUT_PREDICTION_HASH,
        alpha=float(allocation.get("alpha", 0.2)),
        min_calibration_n=int(allocation["min_calibration_n"]),
        embargo_steps=embargo_steps,
        write=True,
    )
    print(
        f"Conformal widths: {widths.height:,} rows over "
        f"{widths['symbol'].n_unique():,} names, embargo {embargo_steps} observation(s)"
    )

result = run_backtest(
    CASE_STUDY_ID,
    HOLDOUT_PREDICTION_HASH,
    spec,
    prices=prices,
    predictions=predictions,
    label=LABEL,
    register=True,
    initial_cash=bt_config.initial_cash,
    calendar=bt_config.calendar,
)
if result.backtest_hash != prospective_hash:
    raise RuntimeError(
        f"the guard predicted {prospective_hash} and the runner registered "
        f"{result.backtest_hash}. The guard decides what may run on the holdout, so a guard "
        "that no longer reproduces the runner's identity is not a smaller problem than the one "
        "it was written for."
    )
print(f"Holdout backtest: {result.backtest_hash}")

# The stage is checked rather than trusted. This configuration comes from the risk stage and its
# spec carries a risk block, and stage inference reads the prediction's split before that block - so
# a holdout run files as `holdout`. If that order ever changes, the whole out-of-sample result lands
# in `risk_overlay` and `20_strategy_analysis` finds no holdout at all, which is a failure four
# notebooks away from its cause.
with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
    registered_stage = conn.execute(
        "SELECT stage FROM backtest_runs WHERE backtest_hash = ?", (result.backtest_hash,)
    ).fetchone()[0]
if registered_stage != "holdout":
    raise RuntimeError(
        f"the holdout backtest registered under stage={registered_stage!r} rather than "
        "'holdout'; the split-based inference in registry.store._infer_stage did not take "
        "precedence over this configuration's risk block"
    )

Prices: 50,200 rows, 100 funds
Predictions: 46,466 rows, 497 sessions
  SKIP backtest (complete (hash=89e706faf2fe)) — reusing cached result
Holdout backtest: 89e706faf2fe


## 4. What it came out at

The two numbers below are one strategy measured on two disjoint periods, and the gap between
them is not an estimate of decay. The validation figure is the maximum of a ranking over more
than a thousand backtests, so it carries the selection; the holdout figure is one measurement
over a much shorter window, so it carries that window's sampling error. Both push the pair
apart on their own, before any real change in the strategy's edge.
[`20_strategy_analysis`](20_strategy_analysis.ipynb) is where they are given intervals and a
paired comparison.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 4. 結果是多少

下方兩個數值是同一策略在兩段互不重疊期間的衡量結果，而兩者差距並不是衰退程度的估計。驗證數值是超過一千次回測排名中的最大值，因此帶有篩選效果；保留測試數值則是在短得多的期間內進行的一次衡量，因此帶有該期間的抽樣誤差。即使策略優勢沒有真正改變，這兩個因素本身也會拉大兩個數值的差距。[`20_strategy_analysis`](20_strategy_analysis.ipynb) 會為兩者提供區間與配對比較。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
metrics = result.metrics
# The selected configuration's own registered Sharpe, not the resolver's. `resolve_solvent_carrier`
# reports the common-support figure, which re-ranks candidates on the timestamps every one of them
# covers; that is the right number for choosing between candidates and the wrong one to set beside a
# holdout measured over its own full window. Both are printed, so neither has to be inferred from
# the other.
with sqlite3.connect(str(CASE_DIR / "run_log" / "registry.db")) as conn:
    carrier_sharpe, carrier_periods, carrier_trades = conn.execute(
        "SELECT sharpe, n_periods, num_trades FROM backtest_metrics WHERE backtest_hash = ?",
        (carrier["val_backtest_hash"],),
    ).fetchone()

print(f"Validation Sharpe over {int(carrier_periods):,} sessions: {carrier_sharpe:.3f}")
print(f"  the same run re-ranked on common support: {carrier['val_sharpe']:.3f}")
print(
    f"Holdout Sharpe over {int(metrics['n_periods']):,} sessions:    "
    f"{metrics.get('sharpe', float('nan')):.3f}"
)
print(
    f"Holdout: CAGR {metrics.get('cagr', float('nan')):.1%}, "
    f"max drawdown {metrics.get('max_drawdown', float('nan')):.2%}, "
    f"win rate {metrics.get('win_rate', float('nan')):.0%}"
)
# This case study runs the bar-by-bar engine - the configuration it overlays declares a trailing
# stop, which a vectorized weight-times-return path cannot express - so trade counts are recorded
# and can be compared. A holdout that rebalanced far less than the validation run at the same
# cadence would say the basket stopped changing, which is a different thing from a lower Sharpe.
print(
    f"Trades: {int(metrics.get('num_trades', 0)):,} on the holdout, "
    f"{int(carrier_trades):,} on validation"
)

Validation Sharpe over 2,011 sessions: 1.023
  the same run re-ranked on common support: 1.028
Holdout Sharpe over 502 sessions:    1.310
Holdout: CAGR 8.7%, max drawdown -5.63%, win rate 74%
Trades: 398 on the holdout, 2,542 on validation


## What this notebook establishes, and what it does not

It establishes a return series for the selected configuration over a period no choice in this
case study was made on. That is the only thing a holdout can give, and it is worth less than
it looks: the window is short beside the validation span it is being compared with, which is
too few observations to separate a strategy that decayed from one that had an ordinary year.

It does not establish that this configuration was the right one to carry here. The selection
that brought it was made on validation, over a pool large enough that its maximum is
optimistic by construction, and this notebook inherits that pool without correcting for it.
The deflation is [`20_strategy_analysis`](20_strategy_analysis.ipynb)'s.

Re-running this notebook against the same configuration is free and idempotent - the backtest hash
is unchanged and the registered run is served back. A different configuration is refused, for the
reason 18 gives.

**Next:** [`20_strategy_analysis`](20_strategy_analysis.ipynb).

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 本 Notebook 確立了什麼，以及尚未確立什麼

它確立選定組態在一段本案例研究未據以做出任何選擇的期間內，產生的報酬率序列。這是保留測試集唯一能提供的事，而其證據力比表面看來更有限：這段期間比拿來比較的驗證期間短，觀測值不足以分辨策略真的衰退，還是只經歷了一個尋常的年度。

它無法確立把這個組態帶到此處是否正確。使它入選的篩選是在驗證資料上進行，候選池大到其最大值在設計上就會偏向樂觀；本 Notebook 承接這個候選池，卻未對其修正。這項校正由 [`20_strategy_analysis`](20_strategy_analysis.ipynb) 處理。

以相同組態重新執行本 Notebook 不增加成本且具冪等性：回測雜湊值不變，已登錄的執行結果會直接傳回。若換成不同組態，則會因第 18 份 Notebook 說明的理由而遭到拒絕。

**下一步：**[`20_strategy_analysis`](20_strategy_analysis.ipynb)。
<!-- ML4T_ZH_END c0012-p000 -->
